# Revenue, AOV & Geographic GMV Analysis

**Business question:** Where is revenue coming from — by time, product category, and geography — and how is average order value evolving?

**Decisions supported:**
- Revenue planning and forecasting
- Category investment prioritisation
- Geographic logistics and marketing resource allocation


## Data Sources

| Query | Description | Grain |
|---|---|---|
| Q1: Monthly revenue & AOV | `vw_order_fact` — delivered orders only | One row per calendar month |
| Q2: Top 10 categories | `vw_item_fact` — delivered orders only | One row per category |
| Q3: GMV by state | `vw_order_fact` — delivered orders only | One row per customer state |

**Key columns used:** `revenue_month`, `total_revenue`, `total_orders`, `average_order_value`, `category_english`, `total_item_value`, `customer_state`, `gmv`, `unique_customers`

**Filter:** All queries restrict to `order_status = 'delivered'`. Cancelled, unavailable, and in-transit orders are excluded.


In [ ]:
import sys
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from IPython.display import display

_REPO_ROOT = Path().resolve().parents[1]
if str(_REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(_REPO_ROOT))

from analysis.utils.db import get_connection
from analysis.utils.sql_loader import get_sql_path, load_queries
from analysis.utils.plotting import apply_style, save_figure

apply_style()

# Load SQL queries
sql_path = get_sql_path("sql/analysis/01_revenue_and_aov_behavior.sql")
queries = load_queries(sql_path)

# Execute all three queries against the database
with get_connection() as conn:
    df_monthly_revenue = pd.read_sql(queries[0], conn)
    df_top_categories  = pd.read_sql(queries[1], conn)
    df_top_states      = pd.read_sql(queries[2], conn)

# Ensure datetime dtype for the time axis
df_monthly_revenue["revenue_month"] = pd.to_datetime(df_monthly_revenue["revenue_month"])

# ---------------------------------------------------------------------------
# Inline data validation
# ---------------------------------------------------------------------------
_checks_01 = [
    ("Monthly revenue rows > 0",         len(df_monthly_revenue) > 0),
    ("No null revenue values",           df_monthly_revenue["total_revenue"].notna().all()),
    ("All revenue values > 0",           (df_monthly_revenue["total_revenue"] > 0).all()),
    ("Top-categories rows > 0",          len(df_top_categories) > 0),
    ("Top-states rows > 0",              len(df_top_states) > 0),
    ("GMV values non-negative",          (df_top_states["gmv"] >= 0).all()),
]
print("Notebook 01 — Data Validation")
for label, passed in _checks_01:
    print(f"  [{'PASS' if passed else 'FAIL'}]  {label}")

print()
display(df_monthly_revenue.head(6))
display(df_top_categories)
display(df_top_states)


## Analytical Methodology

**Methods applied:**
- **Time-series line chart** for revenue trend (panel A): chosen because it preserves the temporal sequence and allows the eye to track directional change.
- **Dual-axis overlay** for AOV on the same time axis: allows simultaneous comparison of two metrics on different scales without a separate chart.
- **Horizontal bar chart** for category ranking (panel B): the horizontal orientation accommodates long category labels without rotation.
- **Bar chart** for order volume seasonality (panel C): discrete monthly bars are more legible than a continuous line for volume counts.
- **Gradient-coloured bar chart** for state GMV (panel D): single-hue gradient encodes magnitude without introducing false categorical distinctions.

**Why this method:** Revenue, AOV, and geographic concentration are descriptive metrics best conveyed by rank-ordered comparisons and time-series trend lines. No statistical modeling is required or appropriate for descriptive GMV analysis.


In [ ]:
# =============================================================================
# Dashboard 01 — Revenue, AOV & Geographic GMV
# =============================================================================
fig = plt.figure(figsize=(18, 14))
fig.suptitle(
    "Olist Revenue & AOV Dashboard",
    fontsize=16, fontweight="bold", y=0.98,
)

# ---------------------------------------------------------------------------
# Panel A (top, wide): Monthly Revenue Trend with AOV overlay
# ---------------------------------------------------------------------------
ax_rev   = fig.add_subplot(3, 2, (1, 2))   # spans both columns

color_rev = "#2196F3"
color_aov = "#FF9800"

ax_rev.fill_between(
    df_monthly_revenue["revenue_month"],
    df_monthly_revenue["total_revenue"],
    alpha=0.18, color=color_rev,
)
ax_rev.plot(
    df_monthly_revenue["revenue_month"],
    df_monthly_revenue["total_revenue"],
    marker="o", color=color_rev, linewidth=2, label="Total Revenue (BRL)",
)
ax_rev.set_title("A  |  Monthly Revenue Trend with Average Order Value (AOV)", loc="left", pad=8)
ax_rev.set_xlabel("Month")
ax_rev.set_ylabel("Total Revenue (BRL)", color=color_rev)
ax_rev.tick_params(axis="y", labelcolor=color_rev)
ax_rev.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))

# Overlay AOV on a secondary y-axis
ax_aov = ax_rev.twinx()
ax_aov.plot(
    df_monthly_revenue["revenue_month"],
    df_monthly_revenue["average_order_value"],
    marker="s", linestyle="--", color=color_aov, linewidth=1.8, label="AOV (BRL)",
)
ax_aov.set_ylabel("Average Order Value (BRL)", color=color_aov)
ax_aov.tick_params(axis="y", labelcolor=color_aov)
ax_aov.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))

# Annotate overall revenue peak
peak_idx = df_monthly_revenue["total_revenue"].idxmax()
peak_row = df_monthly_revenue.loc[peak_idx]
ax_rev.annotate(
    f"Peak: {peak_row['total_revenue']:,.0f}",
    xy=(peak_row["revenue_month"], peak_row["total_revenue"]),
    xytext=(0, 18), textcoords="offset points",
    arrowprops=dict(arrowstyle="->", color="#555"),
    fontsize=9, color="#555",
)

# Combined legend
lines1, labels1 = ax_rev.get_legend_handles_labels()
lines2, labels2 = ax_aov.get_legend_handles_labels()
ax_rev.legend(lines1 + lines2, labels1 + labels2, loc="upper left", fontsize=9)
ax_rev.grid(True, axis="y", linestyle="--", alpha=0.5)
ax_aov.grid(False)

# ---------------------------------------------------------------------------
# Panel B (middle, wide): Top 10 Categories by Revenue — horizontal bar
# ---------------------------------------------------------------------------
ax_cat = fig.add_subplot(3, 2, (3, 4))

df_cat_sorted = df_top_categories.sort_values("total_revenue", ascending=True)
bars = ax_cat.barh(
    df_cat_sorted["category_english"].str.replace("_", " ").str.title(),
    df_cat_sorted["total_revenue"],
    color="#2196F3",
)
ax_cat.set_title("B  |  Top 10 Product Categories by Revenue (Delivered Orders)", loc="left", pad=8)
ax_cat.set_xlabel("Total Revenue (BRL)")
ax_cat.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x/1e6:.1f}M"))

# Annotate the top bar with its revenue value
top_val = df_cat_sorted["total_revenue"].iloc[-1]
ax_cat.text(
    top_val * 0.98,
    len(df_cat_sorted) - 1,
    f"  {top_val/1e6:.2f}M",
    va="center", ha="right", fontsize=9, color="white", fontweight="bold",
)
ax_cat.grid(True, axis="x", linestyle="--", alpha=0.5)
ax_cat.set_axisbelow(True)

# ---------------------------------------------------------------------------
# Panel C (bottom-left): Monthly Order Volume — captures seasonality signal
# ---------------------------------------------------------------------------
ax_vol = fig.add_subplot(3, 2, 5)
ax_vol.bar(
    df_monthly_revenue["revenue_month"],
    df_monthly_revenue["total_orders"],
    color="#9C27B0", width=20,
)
ax_vol.set_title("C  |  Monthly Order Volume", loc="left", pad=8)
ax_vol.set_xlabel("Month")
ax_vol.set_ylabel("Orders")
ax_vol.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))
ax_vol.tick_params(axis="x", rotation=45)
ax_vol.grid(True, axis="y", linestyle="--", alpha=0.5)

# ---------------------------------------------------------------------------
# Panel D (bottom-right): GMV by State — gradient intensity colouring
# ---------------------------------------------------------------------------
ax_state = fig.add_subplot(3, 2, 6)

df_state_sorted = df_top_states.sort_values("gmv", ascending=False).head(10)
gmv_vals = df_state_sorted["gmv"]

# Colour bars by GMV intensity using a single-hue gradient
norm = plt.Normalize(gmv_vals.min(), gmv_vals.max())
colors_state = plt.cm.Blues(norm(gmv_vals) * 0.6 + 0.35)

ax_state.bar(df_state_sorted["customer_state"], gmv_vals, color=colors_state)
ax_state.set_title("D  |  Top 10 States by GMV", loc="left", pad=8)
ax_state.set_xlabel("State")
ax_state.set_ylabel("GMV (BRL)")
ax_state.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x/1e6:.1f}M"))
ax_state.grid(True, axis="y", linestyle="--", alpha=0.5)
ax_state.set_axisbelow(True)

plt.tight_layout(rect=[0, 0, 1, 0.97])
save_figure(fig, "01_revenue_dashboard.png")
plt.show()


# Revenue, AOV & Geographic GMV Analysis — Conclusions

---

## Key Findings
- Total platform revenue shows consistent positive growth over the evaluated timeline.
- A single pronounced revenue peak is observable, interrupting the otherwise steady growth trend.
- Average Order Value (AOV) remains structurally flat despite the significant growth in order volume and total revenue.
- A small cluster of core product categories generates the vast majority of total GMV.
- Geographic GMV generation is heavily concentrated in the top 4 states (SP, RJ, MG, RS).

## Business Implications
- Revenue growth is entirely driven by new customer acquisition and order volume, exposing the platform to acquisition saturation risks.
- The flat AOV indicates that up-selling and cross-selling mechanics are underperforming or non-existent in the purchase flow.
- Geographic revenue concentration amplifies the commercial impact of regional logistics disruptions or competitor campaigns.

## Actionable Recommendations
- Standardize promotional mechanics targeting higher-value bundles or minimum-spend thresholds to artificially lift AOV.
- Prioritize operational and marketing resource allocation toward expanding the customer base in the current top 4 states.
- Monitor category saturation levels within the highest-GMV product sectors to predict impending growth plateaus.
